<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# MarketPulse — Notebook 4 — Dashboard Power BI · Vue Account Manager
### 📝 VERSION APPRENANT

> Ce notebook est un guide pratique Power BI. Les mesures DAX sont à compléter.
> Les `📝 TODO DAX` sont des formules à écrire dans Power BI Desktop.

---
## 0. Préparation des fichiers

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys

pd.set_option('display.max_columns', 50)
COLORS = {'primary':'#534AB7','secondary':'#1D9E75','warning':'#EF9F27','danger':'#E24B4A','neutral':'#888780','light':'#EEEDFE'}
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#F9F9F8','axes.grid':True,'grid.alpha':0.35,'font.size':11})

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/marketpulse/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/marketpulse/data/'
print(f'Configuration OK — {SAVE_PATH}')


In [ ]:
# 📝 TODO : Vérifier les KPIs de référence (valeurs attendues dans Power BI)
# 💡 Indice 1 : Charger marketpulse_analytics.csv
# 💡 Indice 2 : Calculer ROAS, CTR, CPC, nb campagnes, % sous-perf
# 💡 Indice 3 : Ces valeurs seront tes références de validation dans Power BI

df_an = pd.read_csv(analytics_path) if os.path.exists(analytics_path) else pd.read_csv(BASE_URL+'marketpulse_analytics.csv')
df_perf = pd.read_csv(BASE_URL + 'performances_daily.csv')
# TODO : calculer les KPIs de référence
print('=== VALEURS ATTENDUES DANS POWER BI ===')
# ROAS attendu : 3.31× · CTR : 2.86% · Sous-perf : 26.3%


In [ ]:
# Export des fichiers optimisés pour Power BI
import duckdb
conn = duckdb.connect()
df_perf = pd.read_csv(BASE_URL + 'performances_daily.csv', parse_dates=['date_perf'])
df_perf['ctr_recalc']  = (df_perf['clics'] / df_perf['impressions'].replace(0, np.nan)).round(4)
df_perf['roas_recalc'] = np.where((df_perf['revenu_genere_fcfa']>=0)&(df_perf['depenses_fcfa']>0),(df_perf['revenu_genere_fcfa']/df_perf['depenses_fcfa']).round(3),np.nan)
df_perf['is_clean'] = ((df_perf['clics']<=df_perf['impressions'])&(df_perf['revenu_genere_fcfa']>=0)&(df_perf['roas']<=50)&(df_perf['ctr_recalc']<=0.20)).astype(int)
df_perf['date_perf'] = df_perf['date_perf'].dt.strftime('%Y-%m-%d')
df_perf.to_csv(f'{SAVE_PATH}pbi_performances_daily.csv', index=False)
print(f'✅ pbi_performances_daily.csv exporté ({len(df_perf):,} lignes)')


---
## 1. Modèle de données Power BI

### MÉTHODE
Schéma en étoile. Relations à créer dans Power BI Desktop (vue Modèle).

**6 fichiers à connecter :** `pbi_marketpulse_analytics.csv`, `pbi_performances_daily.csv`, `clients_agence.csv`, `audiences.csv`, `contacts_crm.csv`, `campagnes_risque_scores.csv` (NB5)

**Relations :**
| Depuis | Vers | Clé | Cardinalité |
|---|---|---|---|
| `marketpulse_analytics[campagne_id]` | `pbi_performances_daily[campagne_id]` | campagne_id | 1→N |
| `marketpulse_analytics[client_id]` | `clients_agence[client_id]` | client_id | N→1 |
| `Calendrier[Date]` | `pbi_performances_daily[date_perf]` | date | 1→N (active) |
| `Calendrier[Date]` | `marketpulse_analytics[date_debut]` | date | 1→N (inactive) |

---
## 2. Table Calendrier DAX

### MÉTHODE
Créer depuis **Modélisation → Nouvelle table**. Obligatoire pour les analyses temporelles.

### 📝 TODO DAX — Table Calendrier
```dax
Calendrier =
VAR date_min = DATE(2022, 6, 1)
VAR date_max = DATE(2024, 5, 31)
RETURN
ADDCOLUMNS(
    CALENDAR(date_min, date_max),
    "Annee",       YEAR([Date]),
    "Mois",        ___,  -- MONTH([Date])
    "Mois_Nom",    ___,  -- FORMAT([Date], "MMM YYYY")
    "Trimestre",   ___,  -- "Q" & QUARTER([Date])
    "Jour_Nom",    ___,  -- FORMAT([Date], "dddd")
    "Est_Weekend", ___   -- IF(WEEKDAY([Date], 2) >= 6, 1, 0)
)
```

> **Après création :** Trier `Mois_Nom` par `Mois` (Outils de colonne → Trier par colonne).

---
## 3. Les 13 mesures DAX

### MÉTHODE
Créer une **table de mesures vide** `_Mesures` (Modélisation → Entrer des données). Toutes les mesures y sont centralisées.

### 📝 TODO DAX — Mesures 1 à 7 : KPIs principaux

```dax
-- Mesure 1 : ROAS Global
ROAS Global =
DIVIDE(
    SUMX(FILTER(pbi_performances_daily, [is_clean]=1), ___),  -- revenu
    SUMX(FILTER(pbi_performances_daily, [is_clean]=1), ___),  -- dépenses
    0
)
-- ✅ Valeur attendue : 3.31

-- Mesure 2 : ROAS 7J Glissant
ROAS 7J Glissant =
VAR date_max = MAX(Calendrier[Date])
VAR date_debut = date_max - ___  -- 6 jours
RETURN
CALCULATE(
    DIVIDE(SUM(pbi_performances_daily[___]), SUM(pbi_performances_daily[___]), 0),
    pbi_performances_daily[date_perf] >= date_debut,
    pbi_performances_daily[date_perf] <= date_max,
    pbi_performances_daily[is_clean] = 1
)

-- Mesure 3 : CTR Moyen %
CTR Moyen % =
DIVIDE(
    SUMX(FILTER(pbi_performances_daily, [is_clean]=1), [___]),  -- clics
    SUMX(FILTER(pbi_performances_daily, [is_clean]=1), [___]),  -- impressions
    0
) * 100
-- ✅ Valeur attendue : 2.86%

-- Mesures 4 à 7 : CPC, CPA, Budget M FCFA, Revenu M FCFA
-- (Même pattern DIVIDE + SUMX FILTER)
```

### 📝 TODO DAX — Mesures 8 à 10 : Sous-performance

```dax
-- Mesure 8 : Nb Campagnes
Nb Campagnes = COUNTROWS(___)
-- ✅ Valeur attendue : 354

-- Mesure 9 : Nb Campagnes Sous-Perf
Nb Campagnes Sous-Perf =
CALCULATE(
    COUNTROWS(marketpulse_analytics),
    marketpulse_analytics[campagne_sous_performe] = ___
)
-- ✅ Valeur attendue : 93

-- Mesure 10 : % Campagnes Sous-Perf
% Campagnes Sous-Perf =
DIVIDE([___], [___], 0) * 100
-- ✅ Valeur attendue : 26.3%
```

### 📝 TODO DAX — Mesures 11 à 13 : LTV, ROAS J3, Alertes ML

```dax
-- Mesure 11 : LTV Moyenne FCFA
LTV Moyenne FCFA =
AVERAGEX(
    FILTER(contacts_crm, contacts_crm[nb_achats] > 0),
    ___  -- contacts_crm[ltv_fcfa]
)

-- Mesure 12 : ROAS J3 Moyen
ROAS J3 Moyen =
AVERAGEX(
    FILTER(marketpulse_analytics, NOT(ISBLANK([roas_j3]))),
    ___
)

-- Mesure 13 : Nb Alertes ML Critiques
Nb Alertes ML Critiques =
CALCULATE(
    COUNTROWS(campagnes_risque_scores),
    campagnes_risque_scores[niveau_alerte] = ___  -- "Critique"
)
```

### 📝 TODO DAX — Mesures bonus (formatage conditionnel)

```dax
-- Couleur ROAS (pour formatage conditionnel)
Couleur ROAS Canal =
VAR roas_val = [ROAS Global]
RETURN
SWITCH(TRUE(),
    roas_val >= 4.0, ___,   -- "#1D9E75" vert
    roas_val >= 2.5, ___,   -- "#EF9F27" orange
    roas_val >= 1.5, ___,   -- "#E24B4A" rouge
                    ___    -- "#791F1F" rouge foncé
)

-- Statut Client (pour alertes churn)
Statut Client =
VAR delta = [ROAS Client Delta Mensuel]
VAR roas  = [ROAS Global]
RETURN
SWITCH(TRUE(),
    delta < ___ && roas < ___, "🔴 Critique",
    delta < ___,               "🟠 Élevé",
    delta < 0,                 "🟡 Modéré",
                               "🟢 Stable"
)
```

---
## 4. Pages 1 à 5 — Guide de configuration

### Page 1 — Vue Executive
**Question :** Santé globale du portefeuille aujourd'hui ?
- Bandeau alerte (rouge si alertes critiques)
- 4 KPI cards : `[Nb Campagnes]`, `[ROAS Global]`, `[CTR Moyen %]`, `[% Campagnes Sous-Perf]`
- Graphique combiné : barres dépenses mensuelles + courbe `[ROAS 7J Glissant]`
- Panel canal : tableau `[ROAS Global]` par canal avec barres de données
- 3 Slicers : Calendrier[Annee_Mois], clients_agence[nom], canal

### Page 2 — Performance Canaux
**Question :** Sur quel canal investir le budget du mois prochain ?
- Anneau : part du budget par canal
- Tableau : ROAS, CTR, CPC, CPA, % sous-perf, `[Rang ROAS Canal]`
- Heatmap : Matrice jour×canal sur `[ROAS Global]` avec formatage conditionnel dégradé

### Page 3 — Audiences & Créatifs
**Question :** Quelle audience performe le mieux ?
- Matrice : type_audience × canal avec `[ROAS Par Type Audience]`
- Scatter : `[Indice Saturation Moyen]` vs `[ROAS Global]`

### Page 4 — Suivi Clients
**Question :** Qui va nous quitter ?
- Scatter risque : ROAS vs delta ROAS (taille bulle = budget)
- Tableau avec colonne `[Statut Client]` colorée par règles

### Page 5 — Alertes ML
**Question :** Quelles campagnes stopper dès maintenant ?
- Bandeau rouge conditionnel
- 3 KPI cards niveaux (Critique/Élevé/Modéré)
- Tableau `campagnes_risque_scores` avec `score_risque` en barres de données
- Donut répartition niveaux

In [ ]:
# 📝 TODO : Vérifier les KPIs de référence pour la checklist de validation Power BI
# 💡 Indice 1 : Charger et calculer les 6 valeurs clés
# 💡 Indice 2 : Afficher chaque valeur avec la valeur attendue

print('=== CHECKLIST AVANT LIVRAISON POWER BI ===')
df_an = pd.read_csv(analytics_path if os.path.exists(analytics_path) else BASE_URL+'marketpulse_analytics.csv')
# Calculer et afficher :
# ☐ ROAS Global = ? → attendu 3.31×
# ☐ CTR Moyen = ? → attendu 2.86%
# ☐ Nb Campagnes = ? → attendu 354
# ☐ % Sous-Perf = ? → attendu 26.3%
# ☐ ROAS Email ≈ ? → attendu 4.67×
# ☐ ROAS Facebook ≈ ? → attendu 2.51×


---
## Bilan NB4

### Checklist de validation — à compléter

```
☐  ROAS Global         = ___  → attendu : 3.31×
☐  CTR Moyen %         = ___  → attendu : 2.86%
☐  CPC FCFA            = ___  → attendu : ~127 FCFA
☐  Nb Campagnes        = ___  → attendu : 354
☐  % Sous-Perf         = ___  → attendu : 26.3%
☐  Navigation 5 pages  = OK ?
☐  Slicers fonctionnels = OK ?
☐  Bandeau rouge Page 5 = conditionnel ?
```


---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.